# 0410 · Evaluación temporal sin fuga · versión 2 (OE3, OE4, H1, H2)

**Qué hace.** Amplía el cuaderno `0410_EvaluacionTemporal` del asesor **sin modificar sus funciones** (se copian tal cual en la sección «Funciones del asesor»). Agrega lo que el anteproyecto y la realimentación piden y el cuaderno base dejaba pendiente:

| Agregado | Por qué |
| --- | --- |
| Random Forest y XGBoost, además de lineal/logística y LightGBM | OE3 y OE4 comparan los cuatro algoritmos |
| Búsqueda aleatoria acotada de hiperparámetros sobre los pliegues | 4.5.3 y realimentación P4 (20 configuraciones por ensamble, semilla y espacio registrados) |
| Festivos desde `DIM.VW_TIEMPO_CALENDARIO_Y_FESTIVOS` (se detiene si falla) | Realimentación «Festivos» |
| Historia sin futuro por cliente y volumen declarado (nulo, no cero) | 4.3.2 promete variables de cliente; realimentación «Volumen» y «Variables de cliente» |
| Exclusión explícita de entregas sin hora y de última milla negativa | Se auditan y se cuentan, no se borran sin registro |
| Métricas por segmento, error en el 10 % más lento, F1 por umbral | Realimentación P4, métricas fijadas antes de la prueba |
| Calibración isotónica evaluada en validación cruzada por bloques | Realimentación P4 (calibración y umbral se eligen en validación) |
| Importancia por permutación en cada pliegue | Insumo de estabilidad para la interpretación (0420) |
| Registro de la decisión antes de la prueba final, y bloqueo de una segunda ejecución | La prueba final se ejecuta una sola vez |
| Prueba final de ambas tareas con IC por días, referencias y comparación con el modelo lineal | H1 confirmatoria; artefactos para 0420 |
| Manifiesto de la corrida | Reproducibilidad (S5) |

**Regla central (igual que el cuaderno del asesor).** Todo se decide con validación de origen móvil (febrero–junio). La prueba final (julio–agosto) se ejecuta **una sola vez**, al final, y después no se ajusta nada.

**Cómo usarlo.**
1. Revisar la configuración. Para una primera corrida rápida, bajar `N_CONFIGS_BUSQUEDA` a 3 y `B_BOOTSTRAP` a 200.
2. Ejecutar hasta la sección 8 («Decisión»). Revisar las tablas y, si el equipo y el asesor están de acuerdo con la propuesta automática, poner `DECISION_APROBADA = True` y ejecutar esa celda: escribe `P4b_decision.json`.
3. Solo entonces poner `EJECUTAR_PRUEBA_FINAL = True` y ejecutar la sección 9. Si ya existe `P4b_prueba_final_*.csv`, la celda se niega a repetir.

**Contraste de H2.** Si existe la base con clima de 0160, se evalúa automáticamente: «internas» frente a «internas+clima» sobre la misma cohorte (guías con clima completo) y con la misma configuración. Con `REAJUSTAR_CON_CLIMA = True` se agrega la versión reajustada (diseño de tres versiones de la realimentación P5).

**Tiempo de cómputo.** La búsqueda hace `N_CONFIGS_BUSQUEDA × 3 pliegues × 3 ensambles × 2 tareas` ajustes sobre una submuestra de `MAX_FILAS_BUSQUEDA` filas. Con 20 configuraciones y 300.000 filas puede tomar varias horas: conviene correrla como *job* de Databricks.

In [ ]:
# Si faltan librerías:
# %pip install duckdb lightgbm xgboost scikit-learn joblib

In [ ]:
# CONFIGURACIÓN
import os, json, glob, sys, time, datetime, warnings
import numpy as np, pandas as pd, duckdb, joblib
import lightgbm as lgb, xgboost as xgb, sklearn
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import (mean_absolute_error, average_precision_score, roc_auc_score, brier_score_loss,
                             log_loss, precision_score, recall_score, f1_score)
warnings.filterwarnings("ignore", category=UserWarning)

PATH_DATABRICKS = "/Volumes/proyectogrado/source/sourcedb/"
CARPETA_SALIDAS = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_10/"
CARPETA_MODELOS = f"{CARPETA_SALIDAS}MODELOS/"
os.makedirs(CARPETA_MODELOS, exist_ok=True)
RUTA_GOLD_V2       = f"{CARPETA_SALIDAS}GOLD_V2_ENVIO.parquet"                       # 0330
RUTA_GOLD_V2_CLIMA = f"{CARPETA_SALIDAS}GOLD_V2_ENVIO_CLIMA.parquet"                 # 0160; si no existe, no se evalúa H2
RUTA_GUIAS         = f"{PATH_DATABRICKS}LOG_GUIAS/RAW/LOG_GUIAS_BOGOTA.parquet"      # solo para ID_CLIENTE
RUTA_CALENDARIO    = f"{PATH_DATABRICKS}PARAMETRICAS/DIM.VW_TIEMPO_CALENDARIO_Y_FESTIVOS.parquet"
COL_FECHA_CAL, COL_FESTIVO = None, None     # None = detección automática

INICIOS_VALIDACION = ["2026-04-01", "2026-05-01", "2026-06-01"]   # ajustar con el balance por mes de 0330
FIN_DESARROLLO     = "2026-07-01"                                 # desde aquí: prueba final
FIN_PRUEBA         = "2026-09-01"
ALERTAS_POR_DIA    = 500      # ESCENARIO de capacidad de atención, no un dato oficial: definirlo con la empresa
SEMILLA            = 42

USAR_VARIABLES_CLIENTE   = True    # historia sin futuro por ID_CLIENTE (conocido en T0)
EXCLUIR_ENTREGA_SIN_HORA = True    # FECHA_ENTREGA a las 00:00:00: el tiempo calculado no es confiable
MODELOS = ["Lineal", "RandomForest", "XGBoost", "LightGBM"]   # "Lineal" = regresión lineal o logística según la tarea
ENSAMBLES = ["RandomForest", "XGBoost", "LightGBM"]
N_CONFIGS_BUSQUEDA = 20        # configuraciones aleatorias por ensamble y tarea
MAX_FILAS_BUSQUEDA = 300_000   # submuestra de entrenamiento en la búsqueda (no en la evaluación)
REAJUSTAR_CON_CLIMA = False    # True: tercera versión de H2 con hiperparámetros propios
B_BOOTSTRAP = 1000             # remuestreos por días para los IC
N_PERMUTACION = 10_000         # filas de validación para la importancia por permutación en cada pliegue
EJECUTAR_BUSQUEDA = True       # False: usa los parámetros guardados en P4b_mejores_parametros.json

## Funciones del asesor (copiadas sin cambios de `0410_EvaluacionTemporal`)

In [ ]:
# =====================================================================================
# P4 · VARIABLES AGREGADAS SIN FUTURO, PLIEGUES TEMPORALES, UMBRAL Y COMPARACIÓN DE MODELOS
# Ubicación sugerida: 0400, reemplaza target_encode_oof (pos. 19), la selección por test
# (pos. 22 y 26) y el bootstrap de filas (pos. 28).
# =====================================================================================
import numpy as np, pandas as pd
from sklearn.metrics import brier_score_loss, log_loss, average_precision_score

# ---------- 1. Media histórica por grupo usando SOLO desenlaces conocidos antes de T0 ----------
def media_historica_sin_futuro(df, grupo, objetivo, t_pred="t0", t_desenlace="t_etiqueta_reg",
                               suavizado=20, historia_hasta=None):
    """Para cada fila: media suavizada del objetivo en su grupo, con las guías cuyo desenlace
    se conoció ESTRICTAMENTE antes del T0 de la fila. historia_hasta=<corte> congela la historia
    (agregado fijo, como si se hubiera calculado al cierre del entrenamiento); None = se actualiza
    con los desenlaces que van madurando (simulación de uso en línea).
    Devuelve (codificación, n_historia). Sin historia global -> NaN (periodo inicial)."""
    h = df.loc[df[objetivo].notna() & df[t_desenlace].notna(), [grupo, t_desenlace, objetivo]]
    if historia_hasta is not None:
        h = h[h[t_desenlace] < pd.Timestamp(historia_hasta)]
    h = h.sort_values(t_desenlace).rename(columns={t_desenlace: "_t"})
    h["_t"] = h["_t"].astype("datetime64[ns]")                   # misma precisión en ambos lados
    h["_n"] = h.groupby(grupo).cumcount() + 1
    h["_s"] = h.groupby(grupo)[objetivo].cumsum()
    hg = h[["_t", objetivo]].copy()
    hg["_ng"] = np.arange(1, len(hg) + 1); hg["_sg"] = hg[objetivo].cumsum()

    q = df[[grupo, t_pred]].copy(); q["_fila"] = np.arange(len(df))
    q[t_pred] = q[t_pred].astype("datetime64[ns]"); q = q.sort_values(t_pred)
    m = pd.merge_asof(q, h[[grupo, "_t", "_n", "_s"]], left_on=t_pred, right_on="_t", by=grupo,
                      allow_exact_matches=False, direction="backward")
    m = pd.merge_asof(m.sort_values(t_pred), hg[["_t", "_ng", "_sg"]].rename(columns={"_t": "_tg"}),
                      left_on=t_pred, right_on="_tg", allow_exact_matches=False, direction="backward")
    m = m.sort_values("_fila")
    prior = m["_sg"] / m["_ng"]
    n, s = m["_n"].fillna(0), m["_s"].fillna(0)
    enc = (s + suavizado * prior) / (n + suavizado)
    return enc.to_numpy(), n.to_numpy()

# ---------- 2. Pliegues de origen móvil por bloques de calendario ----------
def pliegues_origen_movil(df, inicios_validacion, fin_desarrollo, t_pred="t0",
                          t_etiqueta="t_etiqueta_clf"):
    """inicios_validacion: lista de fechas (inicio de cada bloque de validación).
    Entrena con T0 < inicio y SOLO con etiquetas ya conocidas en el inicio (madurez);
    valida con T0 en [inicio, siguiente inicio). Nunca toca la prueba final (> fin_desarrollo)."""
    limites = [pd.Timestamp(x) for x in inicios_validacion] + [pd.Timestamp(fin_desarrollo)]
    for ini, fin in zip(limites[:-1], limites[1:]):
        tr = (df[t_pred] < ini) & (df[t_etiqueta] < ini)
        va = (df[t_pred] >= ini) & (df[t_pred] < fin)
        yield ini, np.flatnonzero(tr), np.flatnonzero(va)

# ---------- 3. Referencias constantes estimadas SOLO con entrenamiento ----------
def referencias_probabilidad(y_train, y_eval):
    p = float(np.mean(y_train))
    pc = np.full(len(y_eval), p)
    return {"p_constante_train": p,
            "Brier_ref": brier_score_loss(y_eval, pc),
            "LogLoss_ref": log_loss(y_eval, pc, labels=[0, 1]),
            "AP_ref (= prevalencia evaluada)": float(np.mean(y_eval))}

# ---------- 4. Tabla de umbrales en VALIDACIÓN (no en prueba) ----------
def tabla_umbrales(y, p, dia, umbrales=np.round(np.arange(0.05, 0.96, 0.05), 2)):
    filas, n_dias = [], len(np.unique(dia))
    for u in umbrales:
        a = p >= u
        tp = int((a & (y == 1)).sum()); fp = int((a & (y == 0)).sum()); fn = int((~a & (y == 1)).sum())
        filas.append({"umbral": u, "alertas_por_dia": a.sum() / n_dias,
                      "exhaustividad": tp / max(tp + fn, 1), "precision": tp / max(tp + fp, 1),
                      "incumplimientos_omitidos": fn, "falsas_alertas": fp})
    return pd.DataFrame(filas)

def umbral_por_capacidad(p, dia, alertas_dia):
    """Umbral tal que, en promedio, se emitan a lo sumo 'alertas_dia' alertas por día (escenario,
    no costo oficial). Se elige en validación y se congela antes de la prueba final."""
    n_dias = len(np.unique(dia))
    k = int(alertas_dia * n_dias)
    return np.inf if k <= 0 else np.sort(p)[::-1][min(k, len(p)) - 1]

# ---------- 5. Diferencia entre dos modelos con bootstrap por bloques de día ----------
def diferencia_por_bloques(y, pred_a, pred_b, dia, metrica, B=2000, semilla=42):
    """metrica(y, pred) -> float. Remuestrea DÍAS completos (respeta la dependencia dentro de la
    jornada, del manifiesto y de la ruta). Incertidumbre de modelos FIJOS sobre el periodo evaluado."""
    y, pred_a, pred_b, dia = map(np.asarray, (y, pred_a, pred_b, dia))
    dias = np.unique(dia); idx = {d: np.flatnonzero(dia == d) for d in dias}
    rng = np.random.default_rng(semilla)
    obs = metrica(y, pred_a) - metrica(y, pred_b)
    reps = []
    for _ in range(B):
        ii = np.concatenate([idx[d] for d in rng.choice(dias, size=len(dias), replace=True)])
        reps.append(metrica(y[ii], pred_a[ii]) - metrica(y[ii], pred_b[ii]))
    lo, hi = np.percentile(reps, [2.5, 97.5])
    return {"diferencia": obs, "ic95_inf": lo, "ic95_sup": hi, "dias": len(dias)}

## Funciones adicionales (no es necesario modificarlas)

In [ ]:
def cargar_festivos(ruta, col_fecha=None, col_festivo=None):
    """Fechas festivas desde DIM.VW_TIEMPO_CALENDARIO_Y_FESTIVOS. Si no identifica las columnas, se detiene."""
    cal = pd.read_parquet(ruta); cols = list(cal.columns)
    if col_fecha is None:
        cand = [c for c in cols if "FECHA" in c.upper() and pd.api.types.is_datetime64_any_dtype(cal[c])] or \
               [c for c in cols if "FECHA" in c.upper()]
        col_fecha = cand[0] if cand else None
    if col_festivo is None:
        cand = [c for c in cols if "FESTIV" in c.upper()]
        col_festivo = cand[0] if cand else None
    if col_fecha is None or col_festivo is None:
        raise ValueError(f"No se identificaron fecha/festivo. Columnas: {cols}. Escribir COL_FECHA_CAL y COL_FESTIVO.")
    v = cal[col_festivo]
    es = v.astype(str).str.strip().str.upper().isin(["1", "S", "SI", "SÍ", "TRUE", "Y", "X"]) if v.dtype == object \
         else v.fillna(0).astype(float) > 0
    f = set(pd.to_datetime(cal.loc[es, col_fecha]).dt.normalize())
    assert len(f) > 0, "El calendario no marca ningún festivo"
    print(f"Festivos: {len(f)} (columnas '{col_fecha}', '{col_festivo}')")
    return f

# ---------- Modelos ----------
def prep2(numericas, categoricas):
    return ColumnTransformer([
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01))]), categoricas),
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), numericas)])

PARAMS_BASE = {
    "LightGBM":     dict(n_estimators=400, learning_rate=0.05, num_leaves=63, subsample=0.8, subsample_freq=1,
                         colsample_bytree=0.8),
    "XGBoost":      dict(n_estimators=400, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8),
    "RandomForest": dict(n_estimators=200, min_samples_leaf=20, max_features="sqrt", max_samples=0.2),
    "Lineal":       dict()}
ESPACIOS = {   # espacio de búsqueda registrado (realimentación P4)
    "LightGBM": dict(n_estimators=[200, 400, 800], learning_rate=[0.02, 0.05, 0.1], num_leaves=[31, 63, 127, 255],
                     min_child_samples=[20, 50, 100, 200], subsample=[0.6, 0.8, 1.0], colsample_bytree=[0.6, 0.8, 1.0],
                     reg_lambda=[0.0, 1.0, 5.0], objetivo_reg=["regression", "regression_l1", "huber"]),
    "XGBoost":  dict(n_estimators=[200, 400, 800], learning_rate=[0.02, 0.05, 0.1], max_depth=[4, 6, 8, 10],
                     min_child_weight=[1, 5, 10], subsample=[0.6, 0.8, 1.0], colsample_bytree=[0.6, 0.8, 1.0],
                     reg_lambda=[1.0, 5.0, 10.0], objetivo_reg=["reg:squarederror", "reg:absoluteerror", "reg:pseudohubererror"]),
    "RandomForest": dict(n_estimators=[100, 200, 300], max_depth=[None, 12, 20], min_samples_leaf=[1, 5, 20, 50],
                         max_features=["sqrt", 0.3, 0.5], max_samples=[0.1, 0.2, 0.3])}

def muestrear_configs(nombre, n, semilla):
    rng = np.random.default_rng(semilla)
    esp = ESPACIOS[nombre]
    return [{k: v[rng.integers(len(v))] for k, v in esp.items()} for _ in range(n)]

def construir_modelo(tarea, nombre, numericas, categoricas, params=None):
    p = dict(PARAMS_BASE[nombre] if params is None else params)
    obj = p.pop("objetivo_reg", None)
    if nombre == "Lineal":
        m = LogisticRegression(max_iter=1000) if tarea == "clf" else LinearRegression()
    elif nombre == "LightGBM":
        p.setdefault("subsample_freq", 1)
        m = lgb.LGBMClassifier(**p, random_state=SEMILLA, verbose=-1) if tarea == "clf" else \
            lgb.LGBMRegressor(**p, objective=obj or "regression", random_state=SEMILLA, verbose=-1)
    elif nombre == "XGBoost":
        m = xgb.XGBClassifier(**p, tree_method="hist", eval_metric="logloss", random_state=SEMILLA, n_jobs=-1) if tarea == "clf" else \
            xgb.XGBRegressor(**p, tree_method="hist", objective=obj or "reg:squarederror", random_state=SEMILLA, n_jobs=-1)
    elif nombre == "RandomForest":
        m = RandomForestClassifier(**p, random_state=SEMILLA, n_jobs=-1) if tarea == "clf" else \
            RandomForestRegressor(**p, random_state=SEMILLA, n_jobs=-1)
    return Pipeline([("prep", prep2(numericas, categoricas)), ("modelo", m)])

def predecir(m, X, tarea):
    return m.predict_proba(X)[:, 1] if tarea == "clf" else np.maximum(m.predict(X), 0)

# ---------- Métricas ----------
def metricas(tarea, y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    if tarea == "clf":
        return {"n": len(y), "prevalencia": y.mean(), "AP": average_precision_score(y, p), "AUC_ROC": roc_auc_score(y, p),
                "Brier": brier_score_loss(y, p), "LogLoss": log_loss(y, np.clip(p, 1e-6, 1 - 1e-6), labels=[0, 1])}
    e = np.abs(y - p); lento = y >= np.quantile(y, 0.9)
    return {"n": len(y), "MAE_h": e.mean(), "RMSE_h": np.sqrt(np.mean((y - p) ** 2)), "P90_error_abs_h": np.percentile(e, 90),
            "MAE_10pct_mas_lento_h": e[lento].mean(), "MAPE_pct_y>=1h": 100 * np.mean(e[y >= 1] / y[y >= 1]),
            "cobertura_MAPE": np.mean(y >= 1)}

def metricas_por_segmento(pr, seg_cols):
    filas = []
    for tarea in pr.tarea.unique():
        for col in seg_cols:
            sub = pr[pr.tarea == tarea]
            for (modelo, valor), g in sub.groupby(["modelo", col], observed=True):
                if len(g) < 30 or (tarea == "clf" and g.y.nunique() < 2): continue
                m = metricas(tarea, g.y, g.pred)
                filas.append({"tarea": tarea, "segmento": col, "valor": valor, "modelo": modelo, **m})
    return pd.DataFrame(filas)

def calibracion_tabla(y, p, bins=10):
    q = pd.qcut(pd.Series(p).rank(method="first"), bins, labels=False)
    return pd.DataFrame({"y": y, "p": p, "bin": q}).groupby("bin").agg(n=("y", "size"), p_media=("p", "mean"), y_obs=("y", "mean"))

def calibracion_cruzada(pr_modelo):
    """Isotónica ajustada con los bloques ANTERIORES y aplicada al siguiente (sin mirar el bloque evaluado)."""
    bloques = sorted(pr_modelo.bloque.unique()); out = []
    for i in range(1, len(bloques)):
        ant = pr_modelo[pr_modelo.bloque.isin(bloques[:i])]; act = pr_modelo[pr_modelo.bloque == bloques[i]]
        iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(ant.pred, ant.y)
        out.append(act.assign(pred_cal=iso.predict(act.pred)))
    return pd.concat(out) if out else None

## 1. Carga, exclusiones y variables conocidas en T0

In [ ]:
festivos = cargar_festivos(RUTA_CALENDARIO, COL_FECHA_CAL, COL_FESTIVO)
usa_clima = os.path.exists(RUTA_GOLD_V2_CLIMA)
ruta = RUTA_GOLD_V2_CLIMA if usa_clima else RUTA_GOLD_V2
join_cliente = f"""LEFT JOIN (SELECT CAST(NUMERO_GUIA AS BIGINT) AS guia, any_value(ID_CLIENTE) AS ID_CLIENTE
                             FROM read_parquet('{RUTA_GUIAS}') GROUP BY 1) C USING (guia)""" if USAR_VARIABLES_CLIENTE else ""
todo = duckdb.sql(f"SELECT B.*{', C.ID_CLIENTE' if USAR_VARIABLES_CLIENTE else ''} FROM read_parquet('{ruta}') B {join_cliente}").df()
if USAR_VARIABLES_CLIENTE and todo["ID_CLIENTE"].notna().mean() < 0.5:
    print(f"AVISO: ID_CLIENTE está vacío en {todo['ID_CLIENTE'].isna().mean():.0%} de las guías (en el repositorio se dejó en NULL). "
          "Se desactivan las variables de cliente. Para usarlas, ejecutar 0300 versión 6, que lo guarda seudonimizado.")
    USAR_VARIABLES_CLIENTE = False
    todo = todo.drop(columns="ID_CLIENTE")
for c in ["t0", "FECHA_ENVIO", "fecha_entrega", "t_etiqueta_reg", "t_etiqueta_clf"]:
    todo[c] = pd.to_datetime(todo[c])
todo["entrega_sin_hora"] = todo.fecha_entrega.notna() & (todo.fecha_entrega == todo.fecha_entrega.dt.normalize())

pasos = [("Base v2", pd.Series(True, index=todo.index)),
         ("SLA parametrizado", todo.origen_sla == "SLA_PARAMETRIZADO"),
         ("No vencido antes de T0", ~todo.vencido_antes_de_t0.fillna(False).astype(bool))]
if EXCLUIR_ENTREGA_SIN_HORA:
    pasos.append(("Entrega con hora (no 00:00:00)", ~todo.entrega_sin_hora))
if usa_clima:
    pasos.append(("Clima completo (misma cohorte para H1 y H2)", todo.CLIMA_COMPLETO.fillna(False).astype(bool)))
mask, filas = pd.Series(True, index=todo.index), []
for nombre, m in pasos:
    previo = mask.copy(); mask = mask & m
    filas.append({"paso": nombre, "guias": int(mask.sum()), "excluidas_en_el_paso": int((previo & ~m).sum()),
                  "tasa_incumplimiento_excluidas": todo.loc[previo & ~m, "incumplimiento_sla"].mean()})
exclusiones = pd.DataFrame(filas); display(exclusiones)
exclusiones.to_csv(f"{CARPETA_SALIDAS}P4b_exclusiones.csv", index=False)

df = todo[mask].copy()
neg = df.h_ultima_milla < 0
print(f"Última milla negativa con hora: {int(neg.sum())} guías -> objetivo de regresión nulo (se auditan, no se borran)")
df.loc[neg, ["h_ultima_milla", "t_etiqueta_reg"]] = np.nan
df = df.sort_values("t0").reset_index(drop=True)
df["franja"] = pd.cut(df.t0.dt.hour, [-1, 5, 11, 17, 23], labels=["MADRUGADA", "MANANA", "TARDE", "NOCHE"]).astype(str)
df["dia_semana"] = df.t0.dt.dayofweek.astype(str)
df["zona_franja"] = df["categoria_zona"].astype(str) + "_" + df["franja"]
df["dia"] = df.t0.dt.date
df["es_festivo"] = df.t0.dt.normalize().isin(festivos).astype(int)
vol = df.LARGO_ENVIO * df.ANCHO_ENVIO * df.ALTO_ENVIO
df["volumen_declarado_cm3"] = vol.where(vol > 0)          # nulo si falta una dimensión, nunca 0
if USAR_VARIABLES_CLIENTE:
    df["ID_CLIENTE"] = df["ID_CLIENTE"].astype(str)
print(f"{len(df):,} guías; T0 de {df.t0.min()} a {df.t0.max()}")

CATEGORICAS = ["regional_origen", "categoria_zona", "franja", "dia_semana", "ID_TIPOGUIA", "ID_PRODUCTO"]
NUMERICAS = ["sla_h", "margen_sla_en_t0_h", "h_admision_a_t0", "piezas_declaradas", "peso_declarado_kg",
             "volumen_declarado_cm3", "es_festivo", "hist_h_zona_franja", "hist_inc_zona_franja", "n_hist"]
if USAR_VARIABLES_CLIENTE:
    NUMERICAS += ["hist_h_cliente", "hist_inc_cliente", "n_hist_cliente"]
for c in CATEGORICAS: df[c] = df[c].astype(str)
VARIABLES_CLIMA = ["PRECIP_PREV_1H", "PRECIP_MAX_PREV_1H", "LLUVIA_3H_PREV", "LLUVIA_24H_PREV", "LLUVIA_72H_PREV",
                   "TEMP_PREV_1H", "HUM_PREV_1H"]
VERSIONES = {"internas": NUMERICAS}
if usa_clima:
    assert set(VARIABLES_CLIMA) <= set(df.columns), "La base con clima no trae las columnas esperadas"
    VERSIONES["internas+clima"] = NUMERICAS + VARIABLES_CLIMA
    if REAJUSTAR_CON_CLIMA: VERSIONES["internas+clima_reajustada"] = NUMERICAS + VARIABLES_CLIMA
print("Versiones:", list(VERSIONES))
TAREAS = [("clf", "incumplimiento_sla", "t_etiqueta_clf"), ("reg", "h_ultima_milla", "t_etiqueta_reg")]

def agregar_historia2(d, historia_hasta):
    """Agregados sin futuro por zona×franja (función del asesor) y por cliente; congelados en 'historia_hasta'."""
    h = {}
    h["hist_h_zona_franja"], h["n_hist"] = media_historica_sin_futuro(d, "zona_franja", "h_ultima_milla", "t0", "t_etiqueta_reg", historia_hasta=historia_hasta)
    h["hist_inc_zona_franja"], _ = media_historica_sin_futuro(d, "zona_franja", "incumplimiento_sla", "t0", "t_etiqueta_clf", historia_hasta=historia_hasta)
    if USAR_VARIABLES_CLIENTE:
        h["hist_h_cliente"], h["n_hist_cliente"] = media_historica_sin_futuro(d, "ID_CLIENTE", "h_ultima_milla", "t0", "t_etiqueta_reg", historia_hasta=historia_hasta)
        h["hist_inc_cliente"], _ = media_historica_sin_futuro(d, "ID_CLIENTE", "incumplimiento_sla", "t0", "t_etiqueta_clf", historia_hasta=historia_hasta)
    return h

def con_historia(h):
    d = df.copy()
    for k, v in h.items(): d[k] = v
    return d

# Historia de cada pliegue, calculada una sola vez (se guardan solo las columnas nuevas)
HIST = {pd.Timestamp(ini): agregar_historia2(df, ini) for ini in INICIOS_VALIDACION}
PLIEGUES = {t: list(pliegues_origen_movil(df, INICIOS_VALIDACION, FIN_DESARROLLO, t_etiqueta=te)) for t, _, te in TAREAS}
for t, y_col, _ in TAREAS:
    for ini, tr, va in PLIEGUES[t]:
        print(f"{t} · bloque {ini.date()}: entrenamiento {df[y_col].iloc[tr].notna().sum():,}, validación {df[y_col].iloc[va].notna().sum():,}")

## 2. Búsqueda aleatoria de hiperparámetros (solo en validación)

Para cada ensamble y tarea se prueban `N_CONFIGS_BUSQUEDA` configuraciones del espacio `ESPACIOS`. Cada configuración se entrena en cada pliegue con una submuestra aleatoria de hasta `MAX_FILAS_BUSQUEDA` filas de su entrenamiento y se evalúa en el bloque completo de validación. Se elige la de mejor promedio entre bloques: AP para clasificación y MAE para regresión. La prueba final no interviene.

In [ ]:
RUTA_PARAMS = f"{CARPETA_SALIDAS}P4b_mejores_parametros.json"
def busqueda(version_busqueda, numericas):
    filas = []
    for tarea, y_col, _ in TAREAS:
        for nombre in ENSAMBLES:
            configs = muestrear_configs(nombre, N_CONFIGS_BUSQUEDA, SEMILLA + len(nombre))
            for k, cfg in enumerate(configs):
                if tarea == "clf" and "objetivo_reg" in cfg: cfg = {a: b for a, b in cfg.items() if a != "objetivo_reg"}
                vals = []
                for ini, tr, va in PLIEGUES[tarea]:
                    d = con_historia(HIST[ini])
                    tr = tr[d[y_col].iloc[tr].notna().to_numpy()]; va = va[d[y_col].iloc[va].notna().to_numpy()]
                    if len(tr) > MAX_FILAS_BUSQUEDA:
                        tr = np.sort(np.random.default_rng(SEMILLA + k).choice(tr, MAX_FILAS_BUSQUEDA, replace=False))
                    m = construir_modelo(tarea, nombre, numericas, CATEGORICAS, cfg)
                    m.fit(d.iloc[tr][CATEGORICAS + numericas], d.iloc[tr][y_col])
                    p = predecir(m, d.iloc[va][CATEGORICAS + numericas], tarea)
                    vals.append(average_precision_score(d.iloc[va][y_col], p) if tarea == "clf"
                                else mean_absolute_error(d.iloc[va][y_col], p))
                filas.append({"version": version_busqueda, "tarea": tarea, "modelo": nombre, "config": k,
                              "metrica": "AP" if tarea == "clf" else "MAE_h", "promedio": np.mean(vals),
                              "por_bloque": [round(v, 4) for v in vals], **{f"p_{a}": b for a, b in cfg.items()}})
                print(f"{version_busqueda} {tarea} {nombre} config {k}: {np.mean(vals):.4f}")
    return pd.DataFrame(filas)

def mejores(tabla):
    out = {}
    for (v, t, m), g in tabla.groupby(["version", "tarea", "modelo"]):
        g = g.sort_values("promedio", ascending=(t == "reg"))
        fila = g.iloc[0]
        out[f"{v}|{t}|{m}"] = {k[2:]: (None if (isinstance(val, float) and np.isnan(val)) else
                                      (val.item() if hasattr(val, "item") else val))
                               for k, val in fila.items() if k.startswith("p_") and k[2:] in ESPACIOS[m]}
    return out

t_ini = time.time()
if EJECUTAR_BUSQUEDA:
    tablas_b = [busqueda("internas", VERSIONES["internas"])]
    if "internas+clima_reajustada" in VERSIONES:
        tablas_b.append(busqueda("internas+clima_reajustada", VERSIONES["internas+clima_reajustada"]))
    tabla_busqueda = pd.concat(tablas_b, ignore_index=True)
    tabla_busqueda.to_csv(f"{CARPETA_SALIDAS}P4b_busqueda.csv", index=False)
    MEJORES = mejores(tabla_busqueda)
    with open(RUTA_PARAMS, "w") as f: json.dump({"espacios": ESPACIOS, "semilla": SEMILLA, "n_configs": N_CONFIGS_BUSQUEDA,
                                                 "max_filas": MAX_FILAS_BUSQUEDA, "mejores": MEJORES}, f, indent=2, default=str)
else:
    MEJORES = json.load(open(RUTA_PARAMS))["mejores"]
print(f"Búsqueda: {time.time() - t_ini:.0f} s")
display(pd.DataFrame(MEJORES).T)

def params_de(version, tarea, nombre):
    if nombre == "Lineal": return {}
    v = "internas+clima_reajustada" if version == "internas+clima_reajustada" else "internas"   # internas+clima usa la MISMA configuración
    p = MEJORES.get(f"{v}|{tarea}|{nombre}")
    p = dict(PARAMS_BASE[nombre]) if p is None else dict(p)
    for k in ("n_estimators", "num_leaves", "min_child_samples", "min_child_weight", "min_samples_leaf", "max_depth"):
        if k in p and p[k] is not None: p[k] = int(p[k])
    if isinstance(p.get("max_features"), str) and p["max_features"] != "sqrt": p["max_features"] = float(p["max_features"])
    if tarea == "clf": p.pop("objetivo_reg", None)
    return p

## 3. Validación de origen móvil con los cuatro modelos

In [ ]:
pred, imp = [], []
t_ini = time.time()
for version, numericas in VERSIONES.items():
    for tarea, y_col, _ in TAREAS:
        for ini, tr, va in PLIEGUES[tarea]:
            d = con_historia(HIST[ini])
            tr = tr[d[y_col].iloc[tr].notna().to_numpy()]; va = va[d[y_col].iloc[va].notna().to_numpy()]
            Dtr, Dva = d.iloc[tr], d.iloc[va]
            Xtr, Xva = Dtr[CATEGORICAS + numericas], Dva[CATEGORICAS + numericas]
            ytr, yva = Dtr[y_col], Dva[y_col]
            base = pd.DataFrame({"version": version, "tarea": tarea, "bloque": ini.date(), "guia": Dva["guia"].values,
                                 "dia": Dva["dia"].values, "y": yva.values, "categoria_zona": Dva["categoria_zona"].values,
                                 "franja": Dva["franja"].values})
            ref = np.full(len(va), ytr.mean()) if tarea == "clf" else Dva["hist_h_zona_franja"].fillna(ytr.mean()).values
            pred.append(base.assign(modelo="Referencia", pred=ref))
            for nombre in MODELOS:
                m = construir_modelo(tarea, nombre, numericas, CATEGORICAS, params_de(version, tarea, nombre))
                m.fit(Xtr, ytr)
                pred.append(base.assign(modelo=nombre, pred=predecir(m, Xva, tarea)))
                if version == "internas" and nombre in ENSAMBLES:     # estabilidad de importancias (insumo de 0420)
                    s = Xva.sample(min(N_PERMUTACION, len(Xva)), random_state=SEMILLA)
                    r = permutation_importance(m, s, yva.loc[s.index], n_repeats=3, random_state=SEMILLA, n_jobs=1,
                                               scoring="average_precision" if tarea == "clf" else "neg_mean_absolute_error")
                    imp.append(pd.DataFrame({"tarea": tarea, "modelo": nombre, "bloque": ini.date(), "variable": s.columns,
                                             "importancia": r.importances_mean, "desv": r.importances_std}))
            print(f"{version} · {tarea} · {ini.date()} listo ({time.time() - t_ini:.0f} s)")
pred = pd.concat(pred, ignore_index=True)
pred.to_parquet(f"{CARPETA_SALIDAS}P4b_predicciones_validacion.parquet", index=False)
importancias_pliegues = pd.concat(imp, ignore_index=True)
importancias_pliegues.to_csv(f"{CARPETA_SALIDAS}P4b_importancia_pliegues.csv", index=False)

## 4. Tabla de validación, por bloque y por segmento

In [ ]:
tabla = pd.DataFrame([{"version": v, "tarea": t, "modelo": m, **metricas(t, g.y, g.pred)}
                      for (v, t, m), g in pred.groupby(["version", "tarea", "modelo"])])
display(tabla.round(4)); tabla.to_csv(f"{CARPETA_SALIDAS}P4b_tabla_validacion.csv", index=False)
por_bloque = pd.DataFrame([{"version": v, "tarea": t, "modelo": m, "bloque": b, **metricas(t, g.y, g.pred)}
                           for (v, t, m, b), g in pred.groupby(["version", "tarea", "modelo", "bloque"])])
por_bloque.to_csv(f"{CARPETA_SALIDAS}P4b_validacion_por_bloque.csv", index=False)
principal = por_bloque.assign(metrica=np.where(por_bloque.tarea == "clf", por_bloque.AP, por_bloque.MAE_h)) \
                      .pivot_table(index=["version", "tarea", "modelo"], columns="bloque", values="metrica")
print("Métrica principal por bloque (AP en clasificación, MAE en horas en regresión):"); display(principal.round(4))
v = pred[pred.version == "internas"].copy()
v["decil_duracion"] = (pd.qcut(v.y.where(v.tarea == "reg"), 10, labels=False, duplicates="drop") + 1).fillna(0).astype(int)
segmentos = metricas_por_segmento(v, ["categoria_zona", "franja", "decil_duracion"])
segmentos = segmentos[~((segmentos.segmento == "decil_duracion") & (segmentos.tarea == "clf"))]
segmentos.to_csv(f"{CARPETA_SALIDAS}P4b_validacion_por_segmento.csv", index=False)
display(segmentos[segmentos.segmento != "decil_duracion"].round(3).head(40))

## 5. H1 y H2: diferencias con IC 95 % por bloques de día

Cada ensamble frente al modelo lineal de referencia, sobre **las mismas guías**. En AP, una diferencia **positiva** favorece al ensamble; en MAE, una **negativa**. Si el IC no incluye el 0, la diferencia es apreciable en el periodo evaluado.

In [ ]:
def diferencias(pr, version_a, modelo_a, version_b, modelo_b, etiqueta):
    filas = []
    for tarea, metrica in [("clf", average_precision_score), ("reg", mean_absolute_error)]:
        a = pr[(pr.version == version_a) & (pr.tarea == tarea) & (pr.modelo == modelo_a)].sort_values(["bloque", "guia"])
        b = pr[(pr.version == version_b) & (pr.tarea == tarea) & (pr.modelo == modelo_b)].sort_values(["bloque", "guia"])
        assert (a.guia.values == b.guia.values).all(), "Las dos versiones no tienen las mismas guías"
        filas.append({"tarea": tarea, "metrica": "AP" if tarea == "clf" else "MAE_h", "comparacion": etiqueta(tarea),
                      **diferencia_por_bloques(a.y.values, a.pred.values, b.pred.values, a.dia.values, metrica, B=B_BOOTSTRAP, semilla=SEMILLA)})
    return filas
filas = []
for e in ENSAMBLES:
    filas += diferencias(pred, "internas", e, "internas", "Lineal", lambda t, e=e: f"H1 · {e} - {'Logística' if t == 'clf' else 'Lineal'}")
for vclima in [x for x in VERSIONES if x.startswith("internas+clima")]:
    for e in ["LightGBM"]:
        filas += diferencias(pred, vclima, e, "internas", e, lambda t, e=e, vc=vclima: f"H2 · {e} {vc} - internas")
diferencias_val = pd.DataFrame(filas)
diferencias_val["favorece_al_primero"] = np.where(diferencias_val.metrica == "AP", diferencias_val.ic95_inf > 0, diferencias_val.ic95_sup < 0)
display(diferencias_val.round(4)); diferencias_val.to_csv(f"{CARPETA_SALIDAS}P4b_diferencias_validacion.csv", index=False)

## 6. Calibración del clasificador elegido (evaluada sin mirar el bloque evaluado)

In [ ]:
cand_clf = tabla[(tabla.version == "internas") & (tabla.tarea == "clf") & tabla.modelo.isin(MODELOS)].sort_values("AP", ascending=False)
MODELO_CLF_PROPUESTO = cand_clf.iloc[0].modelo
cand_reg = tabla[(tabla.version == "internas") & (tabla.tarea == "reg") & tabla.modelo.isin(MODELOS)].sort_values("MAE_h")
MODELO_REG_PROPUESTO = cand_reg.iloc[0].modelo
ENSAMBLE_H1 = {"clf": cand_clf[cand_clf.modelo.isin(ENSAMBLES)].iloc[0].modelo,
               "reg": cand_reg[cand_reg.modelo.isin(ENSAMBLES)].iloc[0].modelo}
print(f"Mejor en validación · clasificación: {MODELO_CLF_PROPUESTO} · regresión: {MODELO_REG_PROPUESTO}")
print(f"Mejor ensamble (para el contraste confirmatorio de H1) · clasificación: {ENSAMBLE_H1['clf']} · regresión: {ENSAMBLE_H1['reg']}")

pc = pred[(pred.version == "internas") & (pred.tarea == "clf") & (pred.modelo == MODELO_CLF_PROPUESTO)]
cc = calibracion_cruzada(pc)
comp_cal = pd.DataFrame({
    "sin_calibrar": metricas("clf", cc.y, cc.pred),
    "isotonica_cruzada": metricas("clf", cc.y, cc.pred_cal)}).T[["n", "AP", "Brier", "LogLoss"]]
display(comp_cal.round(4)); comp_cal.to_csv(f"{CARPETA_SALIDAS}P4b_calibracion_comparacion.csv")
CALIBRACION_PROPUESTA = "isotonica" if (comp_cal.loc["isotonica_cruzada", "Brier"] < comp_cal.loc["sin_calibrar", "Brier"] and
                                        comp_cal.loc["isotonica_cruzada", "LogLoss"] < comp_cal.loc["sin_calibrar", "LogLoss"]) else "ninguna"
print("Calibración propuesta:", CALIBRACION_PROPUESTA)
calibrador = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(pc.pred, pc.y) if CALIBRACION_PROPUESTA == "isotonica" else None
if calibrador is not None:   # se ajusta con TODAS las predicciones de validación del modelo propuesto
    joblib.dump(calibrador, f"{CARPETA_MODELOS}calibrador_clf_validacion.joblib")
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for nombre, col in [("sin calibrar", "pred"), ("isotónica (cruzada)", "pred_cal")]:
    t = calibracion_tabla(cc.y.values, cc[col].values); ax.plot(t.p_media, t.y_obs, marker="o", label=nombre)
ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_xlabel("Probabilidad predicha"); ax.set_ylabel("Frecuencia observada")
ax.set_title(f"Curva de calibración · {MODELO_CLF_PROPUESTO}"); ax.legend()
plt.tight_layout(); fig.savefig(f"{CARPETA_SALIDAS}P4b_curva_calibracion.png", dpi=130); plt.show()

## 7. Umbral por capacidad de atención (en validación)

In [ ]:
p_val = calibrador.predict(pc.pred) if calibrador is not None else pc.pred.values
umbrales = tabla_umbrales(pc.y.values.astype(int), p_val, pc.dia.values)
umbrales["F1"] = 2 * umbrales.precision * umbrales.exhaustividad / (umbrales.precision + umbrales.exhaustividad).replace(0, np.nan)
display(umbrales.round(3)); umbrales.to_csv(f"{CARPETA_SALIDAS}P4b_tabla_umbrales.csv", index=False)
UMBRAL_PROPUESTO = float(umbral_por_capacidad(p_val, pc.dia.values, ALERTAS_POR_DIA))
a = p_val >= UMBRAL_PROPUESTO
print(f"Escenario de {ALERTAS_POR_DIA} alertas/día -> umbral {UMBRAL_PROPUESTO:.4f}; en validación: "
      f"exhaustividad {recall_score(pc.y, a):.3f}, precisión {precision_score(pc.y, a, zero_division=0):.3f}, "
      f"alertas/día {a.sum() / pc.dia.nunique():.0f}")

## 8. Decisión (antes de la prueba final)

La celda propone la decisión con las reglas fijadas: mejor métrica principal en validación, calibración solo si mejora Brier y pérdida logarítmica en la evaluación cruzada, y umbral por capacidad. **El equipo puede cambiar cualquier valor**, pero solo con argumentos de validación. Al aprobar, se escribe `P4b_decision.json` y ya no se modifica.

In [ ]:
DECISION_APROBADA = False      # cambiar a True cuando el equipo y el asesor estén de acuerdo
DECISION = {
    "modelo_clf": MODELO_CLF_PROPUESTO, "modelo_reg": MODELO_REG_PROPUESTO, "version": "internas",
    "params_clf": params_de("internas", "clf", MODELO_CLF_PROPUESTO), "params_reg": params_de("internas", "reg", MODELO_REG_PROPUESTO),
    "ensamble_h1_clf": ENSAMBLE_H1["clf"], "ensamble_h1_reg": ENSAMBLE_H1["reg"],
    "params_h1_clf": params_de("internas", "clf", ENSAMBLE_H1["clf"]), "params_h1_reg": params_de("internas", "reg", ENSAMBLE_H1["reg"]),
    "calibracion": CALIBRACION_PROPUESTA, "umbral": UMBRAL_PROPUESTO, "alertas_por_dia_escenario": ALERTAS_POR_DIA,
    "variables_categoricas": CATEGORICAS, "variables_numericas": VERSIONES["internas"],
    "excluir_entrega_sin_hora": EXCLUIR_ENTREGA_SIN_HORA, "usar_variables_cliente": USAR_VARIABLES_CLIENTE,
    "fin_desarrollo": FIN_DESARROLLO, "fin_prueba": FIN_PRUEBA, "semilla": SEMILLA}
# Si H2 resultó favorable en validación y el equipo decide usar clima, cambiar "version" y "variables_numericas" aquí.
print(json.dumps(DECISION, indent=2, default=str))
RUTA_DECISION = f"{CARPETA_SALIDAS}P4b_decision.json"
if DECISION_APROBADA:
    if os.path.exists(RUTA_DECISION):
        print("Ya existe una decisión registrada; no se sobrescribe:", RUTA_DECISION)
    else:
        DECISION["fecha_registro"] = datetime.datetime.now().isoformat()
        with open(RUTA_DECISION, "w") as f: json.dump(DECISION, f, indent=2, default=str)
        print("Decisión registrada en", RUTA_DECISION)
else:
    print("Decisión NO registrada (DECISION_APROBADA = False).")

## 9. Prueba final (ejecutar UNA sola vez)

Usa exclusivamente lo registrado en `P4b_decision.json`. Entrena con todo el desarrollo (T0 y etiqueta conocidos antes de `FIN_DESARROLLO`), con la historia congelada en esa fecha, y evalúa julio–agosto. Compara además con el modelo lineal entrenado igual (H1 confirmatoria) y con las referencias. Guarda los modelos y los datos de prueba para `0420` (sin la columna de guía).

In [ ]:
EJECUTAR_PRUEBA_FINAL = False      # cambiar a True UNA sola vez, con la decisión ya registrada
if EJECUTAR_PRUEBA_FINAL:
    assert os.path.exists(RUTA_DECISION), "Primero registrar la decisión (sección 8)"
    assert not glob.glob(f"{CARPETA_SALIDAS}P4b_prueba_final_*.csv"), "La prueba final ya se ejecutó: no se repite"
    DEC = json.load(open(RUTA_DECISION))
    num = DEC["variables_numericas"]; cat = DEC["variables_categoricas"]
    d = con_historia(agregar_historia2(df, FIN_DESARROLLO))
    corte, fin = pd.Timestamp(DEC["fin_desarrollo"]), pd.Timestamp(DEC["fin_prueba"])
    resultados, difs, preds_test = [], [], []
    for tarea, y_col, t_et in TAREAS:
        tr = d.index[(d.t0 < corte) & (d[t_et] < corte) & d[y_col].notna()]
        te = d.index[(d.t0 >= corte) & (d.t0 < fin) & d[y_col].notna()]
        Xtr, ytr, Xte, y = d.loc[tr, cat + num], d.loc[tr, y_col], d.loc[te, cat + num], d.loc[te, y_col].to_numpy()
        nombre = DEC[f"modelo_{tarea}"]
        m = construir_modelo(tarea, nombre, num, cat, DEC[f"params_{tarea}"]).fit(Xtr, ytr)
        p = predecir(m, Xte, tarea)
        lin = construir_modelo(tarea, "Lineal", num, cat).fit(Xtr, ytr); p_lin = predecir(lin, Xte, tarea)
        ens = DEC[f"ensamble_h1_{tarea}"]          # H1 confirmatoria: mejor ensamble de validación frente al lineal, sin calibrar
        p_ens = p if ens == nombre else predecir(construir_modelo(tarea, ens, num, cat, DEC[f"params_h1_{tarea}"]).fit(Xtr, ytr), Xte, tarea)
        joblib.dump(m, f"{CARPETA_MODELOS}modelo_final_{tarea}.joblib")
        fila = {"tarea": tarea, "modelo": nombre, **metricas(tarea, y, p)}
        if tarea == "clf":
            if DEC["calibracion"] == "isotonica":
                assert DEC["modelo_clf"] == MODELO_CLF_PROPUESTO, "El calibrador se ajustó para otro modelo: volver a ejecutar la sección 6 con ese modelo"
                cal = joblib.load(f"{CARPETA_MODELOS}calibrador_clf_validacion.joblib")
                p = cal.predict(p); joblib.dump(cal, f"{CARPETA_MODELOS}calibrador_clf.joblib")
                fila.update({f"{k}_calibrado": v for k, v in metricas("clf", y, p).items() if k in ("Brier", "LogLoss", "AP")})
            y = y.astype(int); a = p >= DEC["umbral"]
            fila.update({f"ref_{k}": v for k, v in referencias_probabilidad(ytr, y).items()})
            fila.update({"umbral": DEC["umbral"], "VP": int((a & (y == 1)).sum()), "FP": int((a & (y == 0)).sum()),
                         "FN": int((~a & (y == 1)).sum()), "VN": int((~a & (y == 0)).sum()),
                         "precision": precision_score(y, a, zero_division=0), "exhaustividad": recall_score(y, a),
                         "F1": f1_score(y, a), "alertas_por_dia": a.sum() / d.loc[te, "dia"].nunique()})
            metrica = average_precision_score
        else:
            ref = d.loc[te, "hist_h_zona_franja"].fillna(ytr.mean()).to_numpy()
            fila.update({"ref_MAE_media_historica_h": mean_absolute_error(y, ref), "ref_MAE_constante_h": mean_absolute_error(y, np.full(len(y), ytr.mean()))})
            metrica = mean_absolute_error
        fila.update({f"lineal_{k}": v for k, v in metricas(tarea, y, p_lin).items() if k in ("AP", "MAE_h", "Brier", "AUC_ROC", "RMSE_h")})
        ic = diferencia_por_bloques(y, p_ens, p_lin, d.loc[te, "dia"].values, metrica, B=B_BOOTSTRAP, semilla=SEMILLA)
        difs.append({"tarea": tarea, "comparacion": f"H1 · {ens} - {'Logística' if tarea == 'clf' else 'Lineal'} (prueba final)",
                     "metrica": "AP" if tarea == "clf" else "MAE_h", **ic})
        resultados.append(fila)
        X_guardar = Xte.copy(); X_guardar["y"] = y; X_guardar["pred"] = p; X_guardar["dia"] = d.loc[te, "dia"].astype(str).values
        X_guardar.to_parquet(f"{CARPETA_MODELOS}prueba_{tarea}.parquet", index=False)            # sin guía
        s = Xtr.sample(min(50_000, len(Xtr)), random_state=SEMILLA); s[y_col] = ytr.loc[s.index]
        s.to_parquet(f"{CARPETA_MODELOS}muestra_entrenamiento_{tarea}.parquet", index=False)    # fondo para SHAP
        preds_test.append(pd.DataFrame({"tarea": tarea, "modelo": nombre, "y": y, "pred": p,
                                        "categoria_zona": d.loc[te, "categoria_zona"].values, "franja": d.loc[te, "franja"].values}))
    final = pd.DataFrame(resultados); display(final.T)
    final.to_csv(f"{CARPETA_SALIDAS}P4b_prueba_final_metricas.csv", index=False)
    difs = pd.DataFrame(difs); display(difs); difs.to_csv(f"{CARPETA_SALIDAS}P4b_prueba_final_diferencias.csv", index=False)
    seg = metricas_por_segmento(pd.concat(preds_test), ["categoria_zona", "franja"])
    seg.to_csv(f"{CARPETA_SALIDAS}P4b_prueba_final_por_segmento.csv", index=False)
else:
    print("Prueba final no ejecutada (EJECUTAR_PRUEBA_FINAL = False).")

## 10. Manifiesto de la corrida

In [ ]:
def huella(patron):
    return [{"archivo": f, "bytes": os.path.getsize(f), "modificado": datetime.datetime.fromtimestamp(os.path.getmtime(f)).isoformat()}
            for f in sorted(glob.glob(patron))]
manifiesto = {"fecha_corrida": datetime.datetime.now().isoformat(), "python": sys.version.split()[0],
              "versiones": {"duckdb": duckdb.__version__, "pandas": pd.__version__, "numpy": np.__version__, "sklearn": sklearn.__version__,
                            "lightgbm": lgb.__version__, "xgboost": xgb.__version__},
              "entradas": {"gold": huella(ruta), "calendario": huella(RUTA_CALENDARIO)},
              "parametros": {k: globals()[k] for k in ["INICIOS_VALIDACION", "FIN_DESARROLLO", "FIN_PRUEBA", "ALERTAS_POR_DIA", "SEMILLA",
                             "USAR_VARIABLES_CLIENTE", "EXCLUIR_ENTREGA_SIN_HORA", "N_CONFIGS_BUSQUEDA", "MAX_FILAS_BUSQUEDA", "B_BOOTSTRAP"]},
              "exclusiones": exclusiones.to_dict("records"), "versiones_evaluadas": list(VERSIONES),
              "salidas": sorted(os.path.basename(f) for f in glob.glob(f"{CARPETA_SALIDAS}P4b_*"))}
with open(f"{CARPETA_SALIDAS}P4b_manifiesto.json", "w") as f: json.dump(manifiesto, f, indent=2, default=str)
print("Manifiesto guardado")

## Qué entregar

**Después de la validación (secciones 1–8):** `P4b_exclusiones.csv`, `P4b_busqueda.csv`, `P4b_mejores_parametros.json`, `P4b_tabla_validacion.csv`, `P4b_validacion_por_bloque.csv`, `P4b_validacion_por_segmento.csv`, `P4b_diferencias_validacion.csv`, `P4b_calibracion_comparacion.csv`, `P4b_curva_calibracion.png`, `P4b_tabla_umbrales.csv` y la decisión `P4b_decision.json`.

**Después de la prueba final (sección 9):** `P4b_prueba_final_metricas.csv`, `P4b_prueba_final_diferencias.csv`, `P4b_prueba_final_por_segmento.csv` y el manifiesto. Con eso se ejecuta `0420_Interpretabilidad`.

**Cómo leer H1.** H1 se sostiene en una tarea si el IC 95 % de la diferencia «ensamble − lineal» excluye el 0 en la dirección favorable, primero en validación y luego, de forma confirmatoria, en la prueba final. Si no, se reporta que no hay evidencia de mejora (H0 no se rechaza).

**Recordatorio.** La prueba final de julio–agosto es la misma que se usó antes para elegir modelos en el 0400 antiguo. Debe declararse en el documento como una evaluación retrospectiva limitada, o bien reservarse datos posteriores a agosto si se vuelven a extraer (realimentación P4).